# Fashion-MNIST 파라미터 실습 — 지난 시간에 배운 것을 직접 돌려 보기

3D어패럴캐드 ch1.4(ANN)에서 배운 **학습률(η) · 히든 레이어 · 문턱값(θ) · 활성화 함수(σ) · 합성곱(CNN)** 을
Fashion-MNIST 데이터로 하나씩 바꿔 보는 노트북입니다. 각 절의 첫 셀에 있는 **대문자 변수**만 고치고,
그 절의 셀을 다시 실행(셀 왼쪽 ▶ 또는 Shift+Enter)하면 됩니다.

- 전체 처음 실행: **런타임 → 모두 실행 (Ctrl+F9)** — CPU 기준 약 8~10분
- 한 절만 다시: 고친 셀부터 **런타임 → 셀 및 하위 셀 실행 (Ctrl+F10)**
- 실행할 때마다 숫자가 조금씩 다른 것은 정상입니다(초기 가중치가 무작위). **경향**을 봅니다.

| 절 | 바꿔 보는 변수 | 지난 시간 슬라이드 |
|---|---|---|
| 1 | `LEARNING_RATES` | 학습률 η (p.13) |
| 2 | `HIDDEN_CONFIGS` | 히든 레이어 삽입 (p.16) |
| 3 | `USE_BIAS`, `CONFIDENCE_THRESHOLDS` | 문턱값 θ (p.17) |
| 4 | `ACTIVATIONS` | 활성화 함수 σ (p.18) |
| 5 | `FILTERS`, `KERNEL_SIZE`, `POOLING` | 합성곱 & 풀링 (p.22~23) |


## 0. 준비 — 데이터와 공용 함수 (그대로 실행)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
print('TensorFlow', tf.__version__)

fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()
train_images = train_images / 255.0
test_images  = test_images  / 255.0
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print('훈련', train_images.shape, '테스트', test_images.shape)

In [ ]:
# 공용 함수: 모델을 만들고(build_fn) → 학습률 lr, epochs 로 학습 → (훈련 정확도 곡선, 테스트 정확도, 걸린 시간)
EPOCHS = 3           # 실험용 반복 횟수 (10이면 결과는 더 좋아지지만 3배 오래 걸림)

def dense_model(hidden=(128,), activation='relu', use_bias=True):
    layers = [tf.keras.Input(shape=(28, 28)), tf.keras.layers.Flatten()]
    for n in hidden:
        layers.append(tf.keras.layers.Dense(n, activation=activation, use_bias=use_bias))
    layers.append(tf.keras.layers.Dense(10, use_bias=use_bias))
    return tf.keras.Sequential(layers)

def train_and_eval(model, lr=0.001, epochs=EPOCHS, verbose=0, seed=1):
    tf.keras.utils.set_random_seed(seed)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=['accuracy'])
    t0 = time.time()
    hist = model.fit(train_images, train_labels, epochs=epochs, verbose=verbose)
    sec = time.time() - t0
    _, test_acc = model.evaluate(test_images, test_labels, verbose=0)
    return hist.history['accuracy'], test_acc, sec

def show_table(rows, header):
    w = [max(len(str(r[i])) for r in [header] + rows) for i in range(len(header))]
    line = lambda r: ' | '.join(str(v).ljust(w[i]) for i, v in enumerate(r))
    print(line(header)); print('-+-'.join('-' * x for x in w))
    for r in rows: print(line(r))

print('준비 완료. EPOCHS =', EPOCHS)

## 1. 학습률(η, learning rate) — 보정 폭을 키우면?

지난 시간(p.13): 오차 ε 가 나오면 가중치를 **η × ε** 만큼만 보정. η 가 크면 빨리 가지만 최적점을 **지나쳐 버린다**.
여기서는 Adam 옵티마이저의 `learning_rate` 가 η 입니다(기본값 0.001).

**바꿔 보기:** `LEARNING_RATES` 목록에 값을 넣고 실행. 0.001을 기준으로 10배씩 키우거나 줄여 보세요.

In [ ]:
LEARNING_RATES = [0.0001, 0.001, 0.01, 0.1, 1.0]     # ← 바꿔 보기

rows = []; curves = {}
for lr in LEARNING_RATES:
    acc_curve, test_acc, sec = train_and_eval(dense_model(), lr=lr)
    curves[lr] = acc_curve
    rows.append([lr, round(acc_curve[-1], 4), round(test_acc, 4), round(sec, 1)])
show_table(rows, ['학습률 η', '훈련 정확도(마지막 epoch)', '테스트 정확도', '시간(초)'])

plt.figure(figsize=(7, 4))
for lr, c in curves.items():
    plt.plot(range(1, len(c) + 1), c, marker='o', label=f'η = {lr}')
plt.xlabel('epoch'); plt.ylabel('train accuracy'); plt.ylim(0, 1); plt.grid(True); plt.legend()
plt.title('learning rate vs. accuracy'); plt.show()

**판정:** 어느 η 에서 정확도가 0.1(= 10개 중 아무거나 찍기) 근처로 무너지나요? 그 이유를 p.13의 그림(최적점을 지나침)으로 설명해 보세요.
너무 작은 η 는 왜 느릴까요?

## 2. 히든 레이어 — 층을 없애거나, 늘리거나

지난 시간(p.16): 히든 레이어가 **많아질수록 표현할 수 있는 비선형성은 커지지만 학습 시간도 는다**.
`HIDDEN_CONFIGS` 의 각 항목은 "히든 레이어 노드 수의 목록"입니다. `()` 는 히든 레이어 없음(입력 784 → 출력 10, 선형 회귀와 같음).

In [ ]:
HIDDEN_CONFIGS = [(), (28,), (128,), (128, 64), (512, 256, 128)]     # ← 바꿔 보기

rows = []
for h in HIDDEN_CONFIGS:
    m = dense_model(hidden=h)
    acc_curve, test_acc, sec = train_and_eval(m)
    rows.append([h if h else '(없음)', m.count_params(), round(acc_curve[-1], 4), round(test_acc, 4), round(sec, 1)])
show_table(rows, ['히든 레이어', '가중치 수', '훈련 정확도', '테스트 정확도', '시간(초)'])

**판정:** 히든 레이어가 없을 때와 하나 있을 때의 차이, 하나에서 셋으로 늘렸을 때의 차이 — 어느 쪽이 더 큰가요?
가중치 수가 10배가 되면 정확도도 10배 좋아지나요?

## 3. 문턱값(θ, threshold) — 뉴런마다 "이 값 이상일 때만"

지난 시간(p.17): 뉴런의 입력 합이 θ 보다 클 때만 다음으로 넘김. Keras 에서는 이것이 **편향(bias)** 입니다.
`y = σ(Σ wx + b)` 에서 `b = −θ`. 차이는 하나 — 지난 시간엔 θ 를 처음에 **랜덤 상수**로 두었지만, Keras 는 θ 도 **학습**합니다.

### 3-1. 편향(θ)을 아예 없애면? — `use_bias=False`

In [ ]:
USE_BIAS = [True, False]      # ← 바꿔 보기 (True = 뉴런마다 θ 있음)

rows = []
for ub in USE_BIAS:
    m = dense_model(hidden=(128,), use_bias=ub)
    acc_curve, test_acc, sec = train_and_eval(m)
    rows.append([ub, m.count_params(), round(acc_curve[-1], 4), round(test_acc, 4)])
show_table(rows, ['use_bias (θ 사용)', '가중치 수', '훈련 정확도', '테스트 정확도'])

# 학습된 θ 들여다보기: 히든 레이어 128개 뉴런의 편향 값 분포
m = dense_model(hidden=(128,)); train_and_eval(m)
b = [l for l in m.layers if isinstance(l, tf.keras.layers.Dense)][0].get_weights()[1]   # 첫 Dense(128) 층의 편향 128개
plt.figure(figsize=(6, 3)); plt.hist(b, bins=30); plt.title('learned bias (= -θ) of 128 hidden neurons'); plt.xlabel('bias'); plt.show()
print('편향 최소/최대/평균:', b.min().round(3), b.max().round(3), b.mean().round(3))

### 3-2. 판정 문턱값 — "확신이 θ 이상일 때만 답한다"

예측 확률의 최댓값이 `θ` 보다 작으면 "모르겠다"로 보류합니다. θ 를 올리면 **답하는 비율(coverage)** 은 줄고 **답한 것 중 정확도**는 오릅니다.

In [ ]:
CONFIDENCE_THRESHOLDS = [0.0, 0.5, 0.7, 0.9, 0.99]     # ← 바꿔 보기

m = dense_model(hidden=(128,)); train_and_eval(m, epochs=EPOCHS)
probs = tf.nn.softmax(m.predict(test_images, verbose=0)).numpy()
pred = probs.argmax(1); conf = probs.max(1)
rows = []
for th in CONFIDENCE_THRESHOLDS:
    answered = conf >= th
    acc_answered = (pred[answered] == test_labels[answered]).mean() if answered.any() else float('nan')
    rows.append([th, round(answered.mean() * 100, 1), round(acc_answered, 4)])
show_table(rows, ['θ (확신 문턱값)', '답한 비율 %', '답한 것 중 정확도'])

**판정:** θ = 0.99 로 두면 몇 %의 이미지만 답하고, 그 정확도는 얼마인가요? 옷 품질 검사 AI라면 θ 를 어디에 두겠습니까?

## 4. 활성화 함수(σ) — sigmoid · tanh · ReLU · 없음

지난 시간(p.18): 활성화 함수는 **비선형**이어야 하고, sigmoid 는 **기울기 소실**, ReLU 는 기울기가 1로 일정.
`None` 은 활성화 함수 없음(선형) — 히든 레이어를 아무리 쌓아도 직선 하나와 같습니다.

In [ ]:
ACTIVATIONS = ['relu', 'sigmoid', 'tanh', None]     # ← 바꿔 보기 ('elu', 'gelu', 'softplus' 등도 가능)
HIDDEN_FOR_ACT = (128, 128, 128)                    # 층을 3개로 두면 sigmoid 의 기울기 소실이 더 잘 보임

rows = []
for a in ACTIVATIONS:
    m = dense_model(hidden=HIDDEN_FOR_ACT, activation=a)
    acc_curve, test_acc, sec = train_and_eval(m)
    rows.append([str(a), round(acc_curve[0], 4), round(acc_curve[-1], 4), round(test_acc, 4), round(sec, 1)])
show_table(rows, ['활성화 함수', '1 epoch 훈련 정확도', '마지막 훈련 정확도', '테스트 정확도', '시간(초)'])

x = np.linspace(-5, 5, 200)
plt.figure(figsize=(7, 3))
plt.plot(x, tf.nn.sigmoid(x), label='sigmoid'); plt.plot(x, tf.nn.tanh(x), label='tanh'); plt.plot(x, tf.nn.relu(x), label='relu'); plt.plot(x, x, '--', label='none (linear)')
plt.ylim(-2, 3); plt.grid(True); plt.legend(); plt.title('activation functions'); plt.show()

**판정:** 활성화 함수가 없을 때(None) 정확도는 히든 레이어가 없을 때(2절의 `()`)와 비슷한가요? 왜 그럴까요?
sigmoid 는 첫 epoch 에서 ReLU 보다 느리게 출발하나요?

## 5. 합성곱(convolution) & 풀링 — 오늘 수업

지난 시간 p.22: **합성곱 = 작은 필터(뿅망치)를 이미지 위에서 밀며 특징 추출**, **풀링 = 데이터 양 줄이기**.
p.23: CNN = [합성곱 → 풀링] 반복 → 펼치기(Flatten) → Dense.

### 5-1. 합성곱을 손으로 — 3×3 필터 하나를 옷 이미지에 밀어 보기

In [ ]:
IMG_INDEX = 0                                   # ← 바꿔 보기: 테스트 이미지 번호
KERNELS = {                                     # ← 바꿔 보기: 3×3 필터 값
    'vertical edge':   [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]],
    'horizontal edge': [[-1, -1, -1], [0, 0, 0], [1, 1, 1]],
    'blur (mean)':     [[1/9] * 3] * 3,
    'sharpen':         [[0, -1, 0], [-1, 5, -1], [0, -1, 0]],
}

img = test_images[IMG_INDEX]
def conv2d_manual(image, kernel):
    k = np.array(kernel, dtype=float); r = k.shape[0] // 2
    H, W = image.shape; out = np.zeros((H - 2 * r, W - 2 * r))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = (image[i:i + 2 * r + 1, j:j + 2 * r + 1] * k).sum()
    return out
def maxpool2x2(a):
    H, W = a.shape[0] // 2 * 2, a.shape[1] // 2 * 2
    return a[:H, :W].reshape(H // 2, 2, W // 2, 2).max(axis=(1, 3))

plt.figure(figsize=(14, 5))
plt.subplot(2, len(KERNELS) + 1, 1); plt.imshow(img, cmap='gray'); plt.title(f'input {img.shape} ({class_names[test_labels[IMG_INDEX]]})'); plt.axis('off')
for n, (name, k) in enumerate(KERNELS.items(), start=2):
    c = conv2d_manual(img, k); p = maxpool2x2(c)
    plt.subplot(2, len(KERNELS) + 1, n); plt.imshow(c, cmap='gray'); plt.title(f'conv: {name} {c.shape}'); plt.axis('off')
    plt.subplot(2, len(KERNELS) + 1, n + len(KERNELS) + 1); plt.imshow(p, cmap='gray'); plt.title(f'+ 2x2 max pooling {p.shape}'); plt.axis('off')
plt.tight_layout(); plt.show()

**판정:** 28×28 이미지에 3×3 필터를 밀면 왜 26×26 이 되나요(p.22의 H−r)? 풀링을 하면 왜 13×13 이 되나요? vertical edge 필터는 어느 부분을 밝게 만드나요?

### 5-2. CNN 만들어 학습 — Dense 모델과 비교

In [ ]:
FILTERS     = 32      # ← 바꿔 보기: 첫 합성곱 층의 필터(뿅망치) 개수
KERNEL_SIZE = 3       # ← 바꿔 보기: 필터 크기 (3, 5, 7)
POOLING     = True    # ← 바꿔 보기: False 면 풀링 없이 (데이터가 안 줄어 더 느림)
CNN_EPOCHS  = 3

def cnn_model(filters=FILTERS, kernel=KERNEL_SIZE, pooling=POOLING):
    L = [tf.keras.Input(shape=(28, 28, 1)),
         tf.keras.layers.Conv2D(filters, kernel, activation='relu', name='conv1')]
    if pooling: L.append(tf.keras.layers.MaxPooling2D(2))
    L.append(tf.keras.layers.Conv2D(filters * 2, kernel, activation='relu', name='conv2'))
    if pooling: L.append(tf.keras.layers.MaxPooling2D(2))
    L += [tf.keras.layers.Flatten(), tf.keras.layers.Dense(64, activation='relu'), tf.keras.layers.Dense(10)]
    return tf.keras.Sequential(L)

cnn = cnn_model(); cnn.summary()
tr = train_images[..., np.newaxis]; te = test_images[..., np.newaxis]
tf.keras.utils.set_random_seed(1)
cnn.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
t0 = time.time(); hist = cnn.fit(tr, train_labels, epochs=CNN_EPOCHS, verbose=1); sec = time.time() - t0
_, cnn_test_acc = cnn.evaluate(te, test_labels, verbose=0)

dense = dense_model(hidden=(128,)); d_curve, dense_test_acc, d_sec = train_and_eval(dense, epochs=CNN_EPOCHS)
show_table([['Dense (784-128-10)', dense.count_params(), round(d_curve[-1], 4), round(dense_test_acc, 4), round(d_sec, 1)],
            [f'CNN ({FILTERS} filters, {KERNEL_SIZE}x{KERNEL_SIZE}, pooling={POOLING})', cnn.count_params(), round(hist.history['accuracy'][-1], 4), round(cnn_test_acc, 4), round(sec, 1)]],
           ['모델', '가중치 수', '훈련 정확도', '테스트 정확도', '시간(초)'])

**판정:** 같은 3 epoch 에서 CNN 이 Dense 보다 정확도가 높은가요? 시간은 몇 배 걸리나요? 필터를 8개로 줄이면, 커널을 7×7 로 키우면 어떻게 되나요?

### 5-3. CNN 이 스스로 배운 필터와, 그 필터가 만든 특징 지도

In [ ]:
SHOW_INDEX = 0                                # ← 바꿔 보기: 특징 지도를 볼 테스트 이미지 번호

w = cnn.get_layer('conv1').get_weights()[0]   # (kernel, kernel, 1, FILTERS)
n = w.shape[-1]; cols = 8
plt.figure(figsize=(cols * 1.2, (n // cols + 1) * 1.2))
for i in range(n):
    plt.subplot(n // cols + 1, cols, i + 1); plt.imshow(w[:, :, 0, i], cmap='gray'); plt.axis('off')
plt.suptitle(f'{n} learned {w.shape[0]}x{w.shape[1]} filters of conv1 (compare with the hand-made edge filters in 5-1)'); plt.show()

feat = tf.keras.Model(cnn.inputs, cnn.get_layer('conv1').output)
fm = feat.predict(te[SHOW_INDEX:SHOW_INDEX + 1], verbose=0)[0]     # (26, 26, FILTERS)
plt.figure(figsize=(cols * 1.2, (n // cols + 1) * 1.2 + 1))
plt.subplot(n // cols + 2, cols, 1); plt.imshow(test_images[SHOW_INDEX], cmap='gray'); plt.title('input'); plt.axis('off')
for i in range(n):
    plt.subplot(n // cols + 2, cols, cols + i + 1); plt.imshow(fm[:, :, i], cmap='gray'); plt.axis('off')
plt.suptitle(f'conv1 feature maps {fm.shape} of test image {SHOW_INDEX} ({class_names[test_labels[SHOW_INDEX]]})'); plt.show()

**판정:** 사람이 만든 edge 필터(5-1)와 CNN 이 학습한 필터(5-3)는 닮았나요? 특징 지도 중 "윤곽"을 잡은 것과 "아무것도 안 잡은" 것을 골라 보세요.

---
### 기록표 (LMS 제출)

| 절 | 바꾼 값 | 내 예상 | 실제 결과 | 한 줄 해석 |
|---|---|---|---|---|
| 1 학습률 | | | | |
| 2 히든 레이어 | | | | |
| 3 문턱값 | | | | |
| 4 활성화 함수 | | | | |
| 5 합성곱 | | | | |

AI 에게 물은 것과 틀렸던 답 한 줄: 